# TF-IDF Model Search

Word-level TF-IDF features for premise/hypothesis pairs with overlap and difference blocks.
Compares vocabulary size, n-gram range, lowercasing, lemmatization (English), stop-word
strategies, and six model families on the same DVC train/validation split.

Target: **70% validation accuracy** (stretch goal; current sparse baselines are ~45%).

In [ ]:
import json
import warnings
from itertools import product
from pathlib import Path

import joblib
import mlflow
import nltk
import pandas as pd
from nltk.stem import WordNetLemmatizer
from scipy.sparse import hstack
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from helpers.data import load_processed_data, load_sample_submission
from helpers.metrics import compute_all_metrics
from helpers.mlflow import log_common_context, log_metrics, log_model_artifacts, start_notebook_run
from helpers.submission import build_submission, save_submission

warnings.filterwarnings('ignore', category=FutureWarning)

## Load Data

In [ ]:
PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
KAGGLE_INPUT_DIR = Path('/kaggle/input/competitions/contradictory-my-dear-watson')

if (PROCESSED_DIR / 'train_split.csv').exists():
    train_df, val_df, test_df = load_processed_data(PROCESSED_DIR)
    sample_submission = load_sample_submission(RAW_DIR / 'sample_submission.csv')
    data_source = 'dvc_processed_split'
elif (KAGGLE_INPUT_DIR / 'train.csv').exists():
    full_train_df = pd.read_csv(KAGGLE_INPUT_DIR / 'train.csv')
    test_df = pd.read_csv(KAGGLE_INPUT_DIR / 'test.csv')
    sample_submission = pd.read_csv(KAGGLE_INPUT_DIR / 'sample_submission.csv')
    train_df, val_df = train_test_split(
        full_train_df,
        test_size=0.2,
        random_state=42,
        stratify=full_train_df['label'],
    )
    data_source = 'kaggle_input_fallback_split'
else:
    raise FileNotFoundError('Could not find DVC processed splits or Kaggle input files.')

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

TARGET_ACCURACY = 0.70
y_train = train_df['label'].astype(int)

print(
    f'Data loaded. Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}, '
    f'source: {data_source}, target accuracy: {TARGET_ACCURACY:.0%}'
)

## Text Preprocessing and TF-IDF Features

Stop-word strategies:
- `keep_all`: no manual removal (TF-IDF still down-weights frequent terms)
- `safe_english`: remove English stop words but keep negation cues (`not`, `no`, ...)
- `vectorizer_english`: sklearn `stop_words='english'` inside `TfidfVectorizer`

Lemmatization uses WordNet for English rows only; other languages keep tokenized text.

In [ ]:
NEGATION_WORDS = {'not', 'no', 'never', 'neither', 'nor', 'none', 'cannot'}
SAFE_ENGLISH_STOP_WORDS = {word for word in ENGLISH_STOP_WORDS if word not in NEGATION_WORDS}


def ensure_nltk_resources() -> None:
    for resource in ('corpora/wordnet', 'corpora/omw-1.4'):
        try:
            nltk.data.find(resource)
        except LookupError:
            package = resource.split('/')[-1]
            nltk.download(package, quiet=True)


ensure_nltk_resources()
LEMMATIZER = WordNetLemmatizer()


def preprocess_text(value, language, feature_config):
    text = str(value)
    if feature_config['lowercase']:
        text = text.lower()
    words = text.split()

    if feature_config['lemmatize'] and language == 'en':
        words = [LEMMATIZER.lemmatize(word) for word in words]

    if feature_config['stop_words'] == 'safe_english' and language == 'en':
        words = [word for word in words if word not in SAFE_ENGLISH_STOP_WORDS]

    return ' '.join(words)


def transform_column(frame, column_name, feature_config):
    return frame.apply(
        lambda row: preprocess_text(row[column_name], row['lang_abv'], feature_config),
        axis=1,
    )


def fit_tfidf_vectorizer(train_frame, feature_config):
    vectorizer_kwargs = {
        'ngram_range': feature_config['ngram_range'],
        'max_features': feature_config['max_features'],
        'lowercase': False,
        'sublinear_tf': True,
    }
    if feature_config['stop_words'] == 'vectorizer_english':
        vectorizer_kwargs['stop_words'] = 'english'

    vectorizer = TfidfVectorizer(**vectorizer_kwargs)
    premise_text = transform_column(train_frame, 'premise', feature_config)
    hypothesis_text = transform_column(train_frame, 'hypothesis', feature_config)
    vectorizer.fit(pd.concat([premise_text, hypothesis_text]))
    return vectorizer


def make_tfidf_features(frame, vectorizer, feature_config):
    premise_text = transform_column(frame, 'premise', feature_config)
    hypothesis_text = transform_column(frame, 'hypothesis', feature_config)

    premise_vec = vectorizer.transform(premise_text)
    hypothesis_vec = vectorizer.transform(hypothesis_text)

    blocks = [premise_vec, hypothesis_vec]
    if feature_config['use_overlap']:
        blocks.append(premise_vec.multiply(hypothesis_vec))
    if feature_config['use_difference']:
        blocks.append(abs(premise_vec - hypothesis_vec))

    return hstack(blocks, format='csr')


def build_feature_matrices(train_frame, val_frame, test_frame, feature_config):
    vectorizer = fit_tfidf_vectorizer(train_frame, feature_config)
    return (
        make_tfidf_features(train_frame, vectorizer, feature_config),
        make_tfidf_features(val_frame, vectorizer, feature_config),
        make_tfidf_features(test_frame, vectorizer, feature_config),
        vectorizer,
    )

## Model Families and Search Grid

In [ ]:
MODEL_FAMILIES = {
    'logistic_regression': lambda: LogisticRegression(
        max_iter=2000,
        random_state=42,
        n_jobs=-1,
    ),
    'linear_svc': lambda: LinearSVC(random_state=42),
    'random_forest': lambda: RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
    'decision_tree': lambda: DecisionTreeClassifier(random_state=42),
    'naive_bayes': lambda: MultinomialNB(),
    'xgboost': lambda: XGBClassifier(
        random_state=42,
        n_estimators=120,
        learning_rate=0.1,
        eval_metric='mlogloss',
        n_jobs=-1,
    ),
}

FEATURE_CONFIGS = [
    {
        'config_id': 'uni_10k_raw',
        'max_features': 10_000,
        'ngram_range': (1, 1),
        'lowercase': True,
        'lemmatize': False,
        'stop_words': 'keep_all',
    },
    {
        'config_id': 'uni_20k_raw',
        'max_features': 20_000,
        'ngram_range': (1, 1),
        'lowercase': True,
        'lemmatize': False,
        'stop_words': 'keep_all',
    },
    {
        'config_id': 'uni_50k_raw',
        'max_features': 50_000,
        'ngram_range': (1, 1),
        'lowercase': True,
        'lemmatize': False,
        'stop_words': 'keep_all',
    },
    {
        'config_id': 'bi_20k_raw',
        'max_features': 20_000,
        'ngram_range': (1, 2),
        'lowercase': True,
        'lemmatize': False,
        'stop_words': 'keep_all',
    },
    {
        'config_id': 'uni_20k_no_lower',
        'max_features': 20_000,
        'ngram_range': (1, 1),
        'lowercase': False,
        'lemmatize': False,
        'stop_words': 'keep_all',
    },
    {
        'config_id': 'uni_20k_lemma',
        'max_features': 20_000,
        'ngram_range': (1, 1),
        'lowercase': True,
        'lemmatize': True,
        'stop_words': 'keep_all',
    },
    {
        'config_id': 'bi_20k_lemma',
        'max_features': 20_000,
        'ngram_range': (1, 2),
        'lowercase': True,
        'lemmatize': True,
        'stop_words': 'keep_all',
    },
    {
        'config_id': 'uni_20k_safe_stop',
        'max_features': 20_000,
        'ngram_range': (1, 1),
        'lowercase': True,
        'lemmatize': False,
        'stop_words': 'safe_english',
    },
    {
        'config_id': 'uni_20k_vectorizer_stop',
        'max_features': 20_000,
        'ngram_range': (1, 1),
        'lowercase': True,
        'lemmatize': False,
        'stop_words': 'vectorizer_english',
    },
    {
        'config_id': 'uni_20k_lemma_safe_stop',
        'max_features': 20_000,
        'ngram_range': (1, 1),
        'lowercase': True,
        'lemmatize': True,
        'stop_words': 'safe_english',
    },
]

for feature_config in FEATURE_CONFIGS:
    feature_config['use_overlap'] = True
    feature_config['use_difference'] = True

SEARCH_PLAN = list(product(FEATURE_CONFIGS, MODEL_FAMILIES.keys()))
print(
    f'{len(FEATURE_CONFIGS)} TF-IDF settings x {len(MODEL_FAMILIES)} model families = '
    f'{len(SEARCH_PLAN)} runs'
)

## Run Grid Search

In [ ]:
feature_cache = {}
grid_results = []

for feature_config, model_family in SEARCH_PLAN:
    config_id = feature_config['config_id']
    run_label = f'{config_id}__{model_family}'
    print(f'Running {run_label}...')

    if config_id not in feature_cache:
        feature_cache[config_id] = build_feature_matrices(
            train_df,
            val_df,
            test_df,
            feature_config,
        )

    x_train, x_val, _, vectorizer = feature_cache[config_id]
    model = MODEL_FAMILIES[model_family]()
    model.fit(x_train, y_train)

    val_preds = model.predict(x_val)
    metrics = compute_all_metrics(val_df, val_preds)

    grid_results.append({
        'run_label': run_label,
        'config_id': config_id,
        'model_family': model_family,
        'max_features': feature_config['max_features'],
        'ngram_range': str(feature_config['ngram_range']),
        'lowercase': feature_config['lowercase'],
        'lemmatize': feature_config['lemmatize'],
        'stop_words': feature_config['stop_words'],
        'accuracy': metrics['accuracy'],
        'f1_macro': metrics['f1_macro'],
        'precision_macro': metrics['precision_macro'],
        'recall_macro': metrics['recall_macro'],
        'meets_target_accuracy': metrics['accuracy'] >= TARGET_ACCURACY,
    })

results_df = pd.DataFrame(grid_results).sort_values(
    ['f1_macro', 'accuracy'],
    ascending=False,
)
results_df.head(15)

## Compare TF-IDF Settings and Stop-Word Strategies

In [ ]:
def best_per_group(frame, group_column):
    ranked = frame.sort_values(['f1_macro', 'accuracy'], ascending=False)
    return ranked.groupby(group_column, as_index=False).first()


tfidf_comparison = best_per_group(results_df, 'config_id')
model_comparison = best_per_group(results_df, 'model_family')
lemmatization_comparison = best_per_group(results_df, 'lemmatize')
stop_word_comparison = best_per_group(results_df, 'stop_words')

print('Best run per TF-IDF config:')
tfidf_comparison[
    ['config_id', 'model_family', 'accuracy', 'f1_macro', 'max_features', 'ngram_range', 'lemmatize', 'stop_words']
]
print('Best run per model family:')
model_comparison[['model_family', 'config_id', 'accuracy', 'f1_macro']]
print('Lemmatized vs non-lemmatized (best run in each group):')
lemmatization_comparison[['lemmatize', 'config_id', 'model_family', 'accuracy', 'f1_macro']]
print('Stop-word strategies (best run in each group):')
stop_word_comparison[['stop_words', 'config_id', 'model_family', 'accuracy', 'f1_macro']]

target_hits = results_df[results_df['meets_target_accuracy']]
print(f"Runs meeting {TARGET_ACCURACY:.0%} accuracy: {len(target_hits)} / {len(results_df)}")

## Select Production Setup and Log to MLflow

In [ ]:
best_result = results_df.iloc[0]
best_config_id = best_result['config_id']
best_model_family = best_result['model_family']
best_feature_config = next(item for item in FEATURE_CONFIGS if item['config_id'] == best_config_id)

RUN_NAME = f"tfidf_production_{best_config_id}__{best_model_family}"
print('Selected production setup:')
print(best_result[['run_label', 'accuracy', 'f1_macro', 'precision_macro', 'recall_macro']])

x_train, x_val, x_test, vectorizer = feature_cache[best_config_id]
production_model = MODEL_FAMILIES[best_model_family]()
production_model.fit(x_train, y_train)

val_preds = production_model.predict(x_val)
production_metrics = compute_all_metrics(val_df, val_preds)
test_preds = production_model.predict(x_test)

submission = build_submission(sample_submission, test_preds)
submission_path = Path('../submissions') / f'{RUN_NAME}.csv'
save_submission(submission, submission_path)

results_path = Path('../submissions') / 'tfidf_grid_search_results.csv'
results_df.to_csv(results_path, index=False)

model_path = Path('../submissions') / f'{RUN_NAME}_model.joblib'
vectorizer_path = Path('../submissions') / f'{RUN_NAME}_vectorizer.joblib'
feature_config_path = Path('../submissions') / f'{RUN_NAME}_feature_config.json'

joblib.dump(production_model, model_path)
joblib.dump(vectorizer, vectorizer_path)
feature_config_path.write_text(
    json.dumps(
        {
            **best_feature_config,
            'ngram_range': list(best_feature_config['ngram_range']),
            'model_family': best_model_family,
        },
        indent=2,
    ),
    encoding='utf-8',
)

serving_type = 'sklearn_engineered_tfidf_text_pair_features'
metadata = {
    'run_name': RUN_NAME,
    'data_source': data_source,
    'serving_type': serving_type,
    'label_mapping': {0: 'entailment', 1: 'neutral', 2: 'contradiction'},
    'model_family': best_model_family,
    'feature_config': {
        **best_feature_config,
        'ngram_range': list(best_feature_config['ngram_range']),
    },
    'selection_metric': 'f1_macro',
    'target_accuracy': TARGET_ACCURACY,
    'meets_target_accuracy': bool(production_metrics['accuracy'] >= TARGET_ACCURACY),
    'feature_order': ['premise_tfidf', 'hypothesis_tfidf', 'overlap_tfidf', 'difference_tfidf'],
    'grid_runs': len(results_df),
    'best_grid_result': best_result.to_dict(),
}

mlflow_params = {
    'model_family': best_model_family,
    'config_id': best_config_id,
    'max_features': best_feature_config['max_features'],
    'ngram_range': str(best_feature_config['ngram_range']),
    'lowercase': best_feature_config['lowercase'],
    'lemmatize': best_feature_config['lemmatize'],
    'stop_words': best_feature_config['stop_words'],
    'use_overlap': best_feature_config['use_overlap'],
    'use_difference': best_feature_config['use_difference'],
    'data_source': data_source,
    'serving_type': serving_type,
    'grid_runs': len(results_df),
    'target_accuracy': TARGET_ACCURACY,
}

with start_notebook_run(
    RUN_NAME,
    tags={
        'model_family': best_model_family,
        'features': 'tfidf_engineered_pair',
        'serving_type': serving_type,
        'stage': 'production_candidate',
    },
):
    mlflow.log_params(mlflow_params)
    log_metrics(production_metrics)
    log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
    mlflow.log_artifact(str(submission_path), artifact_path='submissions')
    mlflow.log_artifact(str(results_path), artifact_path='research')
    log_model_artifacts(
        artifacts={
            'model.joblib': model_path,
            'vectorizer.joblib': vectorizer_path,
            'feature_config.json': feature_config_path,
        },
        metadata=metadata,
        artifact_path='model',
    )

production_summary = {
    'run_name': RUN_NAME,
    'config_id': best_config_id,
    'model_family': best_model_family,
    'accuracy': production_metrics['accuracy'],
    'f1_macro': production_metrics['f1_macro'],
    'target_accuracy': TARGET_ACCURACY,
    'meets_target_accuracy': production_metrics['accuracy'] >= TARGET_ACCURACY,
    'submission_path': str(submission_path),
    'results_table_path': str(results_path),
    'serving_type': serving_type,
}

pd.Series(production_summary)